# Imports

In [ ]:
import numpy as np
import scipy.stats as sp
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import inspect

# Constants

In [ ]:
INITIAL_SEED = 67
N = 10 ** 6 #amount of numbers generated

SEED_2_SHIFT = 1 #increase seed for prng_2 by this to avoid hidden effects based on the seed being the same
HISTOGRAM_BINS = int(np.sqrt(N))
ALPHA = 0.5 #alpha default value for weighted mixing
SAMPLE_SEEDS = range(11, 25 + 1) #seeds for testing across different initial seeds

# File Structure

In [ ]:
FIGURES = "figures"
TABLES = "tables"
RNGDATA = "rngdata"

os.makedirs(FIGURES, exist_ok=True)
os.makedirs(TABLES, exist_ok=True)
os.makedirs(RNGDATA, exist_ok=True)

def make_filename(prng_1, prng_2, hybrid_alg_str, halg_possible_names, seed, alpha):
    filename = "dat_"
    if hybrid_alg_str not in halg_possible_names and hybrid_alg_str != "":
        filename += "non"
        return filename
    filename += str(type(prng_1))[17:-2] + "_"
    if hybrid_alg_str in halg_possible_names:
        filename += str(type(prng_2))[17:-2] + "_"
        filename += hybrid_alg_str + "_"
        if hybrid_alg_str == "wmx": filename += "a" + str(alpha) + "_"
    else:
        filename += "none_"
    filename += "seed" + str(seed)
    return filename

def save_sequence(sequence, filename, silent=False, overwrite=False):
    pathname = RNGDATA + "/" + filename
    if os.path.exists(pathname) and not overwrite:
        if not silent: print("File Already Exists:", filename, end="")
        return
    sequence.tofile(pathname)
    if not silent: print("Saved File:", filename, end="")

def load_sequence(filename, silent=False):
    sequence = np.fromfile(RNGDATA + "/" + filename, dtype=np.uint32)
    if not silent: print("Loaded File:", filename, end="")
    return sequence

# Base PRNG Algorithms

In [ ]:
class MT: #Mersenne Twister (MT19937)
    def __init__(self, seed):
        self.rng = np.random.Generator(np.random.MT19937(seed))
    def next(self):
        return self.rng.integers(0, 2**32, dtype=np.uint32)

class XS: #XOR Shift (XorShift32)
    def __init__(self, seed):
        self.rng = np.uint32(seed)
    def next(self):
        x = self.rng
        x ^= (x << 13) & np.uint32(0xffffffff)
        x ^= (x >> 17)
        x ^= (x << 5) & np.uint32(0xffffffff)
        self.rng = np.uint32(x)
        return self.rng

class PCG: #Permuted Congruential Generator (PCG64)
    def __init__(self, seed):
        self.rng = np.random.Generator(np.random.PCG64(seed))
    def next(self):
        return self.rng.integers(0, 2**32, dtype=np.uint32)

def gen_list(prng, length):
    if isinstance(prng, (MT, PCG)): #if there's implementation available
        return prng.rng.integers(0, 2**32, size=length, dtype=np.uint32) #use fastest option
    seq = np.empty(length, dtype=np.uint32)
    next_cache = prng.next #otherwise, cache the function and do it manually
    for i in range(length):
        seq[i] = next_cache()
    return seq

# Hybridization Algorithms

In [ ]:
def sequential_chaining(prng_1, prng_2, length):
    seed_2 = prng_1.next()
    prng_2 = type(prng_2)(seed_2)
    sequence = gen_list(prng_2, length)
    return sequence

def xor_combination(prng_1, prng_2, length):
    sequence = np.empty(length, dtype=np.uint32)
    for i in range(length):
        x = prng_1.next()
        y = prng_2.next()
        sequence[i] = x ^ y
    return sequence

#impure function due to dependency on ALPHA value; needed to have same amount of necessary arguments as other functions
def weighted_mixing(prng_1, prng_2, length, alpha=ALPHA):
    sequence = np.empty(length, dtype=np.uint32)
    alpha = np.clip(alpha, 0.0, 1.0)
    mask = np.uint64(2**32) #forcing uint64 arithmetic to handle integer alpha properly
    a = np.uint64(round(alpha * mask)) #new integer alpha for keeping integer arithmetic
    z = np.uint64(mask - a)
    for i in range(length):
        x = np.uint64(prng_1.next())
        y = np.uint64(prng_2.next())
        hybrid = (x*a + y*z) >> np.uint64(32) #calculating hybrid number with uint64 everything, then rightshifting back to 32 bits
        sequence[i] = np.uint32(hybrid)
    return sequence

def conditional_interleaving(prng_1, prng_2, length):
    sequence = np.empty(length, dtype=np.uint32)
    for i in range(length):
        x = prng_1.next()
        y = prng_2.next()
        sequence[i] = x if ((x & 1) != (y & 1)) else y
    return sequence

halgs = {
    "chn":sequential_chaining,
    "xor":xor_combination,
    "wmx":weighted_mixing,
    "ilv":conditional_interleaving
}

# Metrics

In [ ]:
def get_repetition_distance(sequence, max_check=10**6):
    seen = {}
    for i, val in enumerate(sequence[:max_check]):
        if val in seen:
            return i - seen[val]
        seen[val] = i
    return float("inf")

def base_stats(sequence):
    seq = np.array(sequence, dtype=np.float64)
    return {
        "size":len(seq),
        "mean":np.mean(seq),
        "median":np.median(seq),
        "variance":np.var(seq),
        "standard deviation":np.std(seq),
        "minimum":np.min(seq),
        "maximum":np.max(seq)
    }

def histogram(sequence, bins, normalize=True):
    if normalize:
        seq = sequence.astype(np.float64) / 2**32
        range_max = 1
    else:
        seq = sequence
        range_max = 2**32
    hist, edges = np.histogram(seq, bins=bins, range=(0, range_max))
    return hist, edges

def shannon_entropy(hist):
    p = hist / np.sum(hist) #probabilities
    p = p[p > 0] #remove zeros
    return sp.entropy(p, base=2) #entropy in bits (base 2), not nats (base e)

def chi_squared_test(hist):
    expected = np.full_like(hist, np.sum(hist) / len(hist), dtype=np.float64)
    chi2, p = sp.chisquare(hist, expected)
    return {"chi2":chi2, "p value":p}

def autocorrelation(sequence, lag=1):
    if lag >= len(sequence):
        return 0
    seq = np.asarray(sequence, dtype=np.float64)
    seq -= np.mean(seq)
    denominator = np.correlate(seq, seq)[0]
    if denominator == 0:
        return 0
    return np.correlate(seq[:-lag], seq[lag:])[0] / denominator

def bit_frequency(sequence):
    seq = np.asarray(sequence, dtype=np.uint32)
    bits = np.unpackbits(seq.view(np.uint8))
    return np.mean(bits)

def bit_autocorrelation(sequence, lag=1):
    seq = np.asarray(sequence, dtype=np.uint32)
    bits = np.unpackbits(seq.view(np.uint8)).astype(np.float64)
    bits -= np.mean(bits)
    return np.correlate(bits[:-lag], bits[lag:])[0] / np.correlate(bits, bits)[0]

def ks_test(sequence): #Kolmogorov-Smirnov test
    norm = np.asarray(sequence, dtype=np.float64) / 2**32
    d, p = sp.kstest(norm, 'uniform')
    return {"D": d, "p value": p}

def approximate_pi(sequence): #for calculating the effectiveness of Monte Carlo methods
    space = (sequence.astype(np.float64) / 2**31) - 1 #points from -1 to 1
    if len(space) % 2 == 1:
        space = space[:-1] #need to cut out last number to ensure equal amount of numbers for x and y coordinates
    x = space[0::2] #odd values will be x coordinates
    y = space[1::2] #even values will be y coordinates
    in_circle = (x**2 + y**2) <= 1 #if said point is inside the circle (radius 1, center (0, 0))
    return 4 * np.mean(in_circle) #ratio of points in circle over points in square (or in 'space' array) yields Pi / 4

# Shortcut Function For Generating Sequences

In [ ]:
def generate_sequence(prng_1_class, prng_2_class, hybrid_alg_str, alpha=ALPHA, seed=INITIAL_SEED):
    prng_1 = prng_1_class(seed)
    prng_2 = prng_2_class(seed + SEED_2_SHIFT)
    hybrid_alg = hybrid_alg_str #chn / xor / wmx / ilv
    
    if hybrid_alg in halgs:
        if hybrid_alg != "wmx": seq = halgs[hybrid_alg](prng_1, prng_2, N)
        else: seq = halgs[hybrid_alg](prng_1, prng_2, N, alpha)
    elif hybrid_alg == "": seq = gen_list(prng_1, N) #non hybrid sequence
    else: seq = None
    
    filename = make_filename(prng_1, prng_2, hybrid_alg, halgs.keys(), seed, alpha)
    return seq, filename

overwrite_files = False
write_output = True

# Initiating & Saving Sequences for RQs all but 4; RQ 5

In [ ]:
rngs = [MT, XS, PCG]
alpha_values = [0.05, 0.25, 0.5, 0.75, 0.95]
counter = 1
times = []
for rng1 in rngs: #generating all single-prng sequences
    timer = time.time()
    seq, filename = generate_sequence(rng1, XS, "")
    save_sequence(seq, filename, overwrite=overwrite_files, silent=not write_output)
    timer = time.time() - timer; throughput = N/timer if timer != 0 else float('inf')
    times.append({"name":filename, "time":timer, "time per number":timer/N, "throughput":throughput})
    if write_output: print(" ({:.2f} seconds)".format(timer), sep=" ")
for rng1 in rngs: #generating all hybrid prng sequences
    for rng2 in rngs:
        for halg in list(halgs.keys()):
            for a in alpha_values: #if doing weighted mixing - apply different values
                timer = time.time()
                seq, filename = generate_sequence(rng1, rng2, halg, a)
                if write_output: print("{:2d}/72".format(counter), end=" ")
                save_sequence(seq, filename, overwrite=overwrite_files, silent=not write_output)
                timer = time.time() - timer; throughput = N/timer if timer != 0 else float('inf')
                times.append({"name":filename, "time":timer, "time per number":timer/N, "throughput":throughput})
                if write_output: print(" ({:.2f} seconds)".format(timer), sep=" "); counter += 1
                if halg != "wmx": #to not iterate needlessly through all alpha variations
                    break

df = pd.DataFrame(times)
df = df.sort_values(by="time", ascending=True)
l = list(df["time"])
time_diffs = [0]
for i in range(1, len(l)):
    time_diffs.append(l[i] - l[i-1])
df["time difference with previous"] = time_diffs
df.to_csv(TABLES + "/generation_times.csv", index=False)

# Initiating & Saving Sequences for RQ 4

In [ ]:
counter = 1

SAMPLE_COMBINATIONS = [ #combinations for testing across different initial seeds
    [MT, XS, "ilv"],
    [XS, MT, "wmx", 0.05],
    [XS, PCG, "xor"],
    [XS, PCG, "wmx", 0.5],
    [MT, PCG, "chn"],
    [PCG, MT, "chn"],
    [MT, PCG, "xor"]
]

for seed in SAMPLE_SEEDS:
    for combo in SAMPLE_COMBINATIONS:
        a = 0 if len(combo) == 3 else combo[3] #default to alpha == 0 if combo isn't wmx
        seq, _ = generate_sequence(combo[0], combo[1], combo[2], alpha=a, seed=seed)
        if write_output: print("{:2d}/105".format(counter), end=" ")
        save_sequence(seq, "dat_sample"+str(SAMPLE_COMBINATIONS.index(combo)+1)+"_seed"+str(seed), overwrite=overwrite_files, silent=not write_output)
        if write_output: print()
        counter += 1

# Loading Sequences

In [62]:
dat_files = os.listdir(RNGDATA)
dat_files_main = [d for d in dat_files if "sample" not in d]
dat_files_samples = [d for d in dat_files if "sample" in d]

# RQ 1, 3, 6, 7

In [63]:
results = []
for f in dat_files_main:
    seq = load_sequence(f, silent=True)
    hist, _ = histogram(seq, HISTOGRAM_BINS)
    chi2 = chi_squared_test(hist); chi2_v, chi2_p = chi2["chi2"], chi2["p value"]
    ks = ks_test(seq); ks_d, ks_p = ks["D"], ks["p value"]
    pi_approx = approximate_pi(seq)
    res = {"name":f} | base_stats(seq)
    res = res | {
        "repetition distance":get_repetition_distance(seq, max_check=len(seq)),
        "entropy":shannon_entropy(hist),
        "chi-square test chi value":chi2_v,
        "chi-square test p value":chi2_p,
        "kolmogorov-smirnov test d value":ks_d,
        "kolmogorov-smirnov test p value":ks_p,
        "autocorrelation (lag 1)":autocorrelation(seq, lag=1),
        "autocorrelation (lag 50)":autocorrelation(seq, lag=50),
        "autocorrelation (lag 200)":autocorrelation(seq, lag=200),
        "bit autocorrelation (lag 1)":bit_autocorrelation(seq, lag=1),
        "bit autocorrelation (lag 15)":bit_autocorrelation(seq, lag=10),
        "bit autocorrelation (lag 50)":bit_autocorrelation(seq, lag=50),
        "bit frequency ratio":bit_frequency(seq),
        "pi approximation (monte carlo)":pi_approx,
        "difference with actual pi value":abs(pi_approx - np.pi)
    }
    results.append(res)

df = pd.DataFrame(results)
df.to_csv(TABLES + "/sequences_stats.csv")

# RQ 2

In [64]:
def extract_hybrid_alg(name):
    if "xor" in name: return "xor"
    if "wmx" in name: return "wmx"
    if "chn" in name: return "chn"
    if "ilv" in name: return "ilv"
    return "none"

df["hybrid type"] = df["name"].apply(extract_hybrid_alg)
summary = df.groupby("hybrid type").mean(numeric_only=True)
summary.to_csv(TABLES + "/sequences_groupedby_hybrid.csv")

# RQ 4

In [65]:
def extract_sample_number(name):
    return name[name.find("sample")+6:name.find("_", 4)]

results = []
for f in dat_files_samples:
    seq = load_sequence(f, silent=True)
    hist, _ = histogram(seq, HISTOGRAM_BINS)
    chi2 = chi_squared_test(hist)
    ks = ks_test(seq)
    res = {"name":f} | base_stats(seq)
    res = res | {
        "entropy":shannon_entropy(hist),
        "chi-square test p value":chi2["p value"],
        "kolmogorov-smirnov test p value":ks["p value"]
    }
    results.append(res)

df = pd.DataFrame(results)
df.to_csv(TABLES + "/samples_stats.csv")

df["sample #"] = df["name"].apply(extract_sample_number)
summary = df.select_dtypes(include="number").groupby(df["sample #"]).agg(["std", "mean", "min", "max", ("range", lambda x: x.max() - x.min())])
summary.to_csv(TABLES + "/samples_groupedby_sample.csv")

# Creating Figures

In [77]:
plt.rcParams.update({'font.size': 7})
show_figures = False

#in case some figures treat numerical values from csv files as strings
#this happens because the floats are written with comma instead of point
#to fix this, add these lines after reading the values from the table:
#values = [str(i).replace(",", ".") for i in values] #changing format to python compatible floats
#values = [float(i) for i in values] #converting strings to floats

Distribution of Values for a Hybrid PRNG, Achieved by Combining MT with PCG Using "XOR Combination" Hybridization Algorithm

In [78]:
seq = load_sequence("dat_MT_PCG_xor_seed" + str(INITIAL_SEED), silent=True)
hist, edges = histogram(seq, HISTOGRAM_BINS)
centers = (edges[:-1] + edges[1:]) / 2 #taking middles between each 2 edges
plt.bar(centers, hist, width=(edges[1] - edges[0]))
plt.xlabel("Value (Normalized)")
plt.ylabel("Frequency")

plt.savefig(FIGURES + "/fig1.1.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Distribution of Values for a Hybrid PRNG, Achieved by Combining MT with PCG Using "Weighted Mixing" Hybridization Algorithm

In [79]:
seq = load_sequence("dat_MT_PCG_wmx_a0.5_seed" + str(INITIAL_SEED), silent=True)
hist, edges = histogram(seq, HISTOGRAM_BINS)
centers = (edges[:-1] + edges[1:]) / 2 #taking middles between each 2 edges
plt.bar(centers, hist, width=(edges[1] - edges[0]))
plt.xlabel("Value (Normalized)")
plt.ylabel("Frequency")

plt.savefig(FIGURES + "/fig1.2.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Mean Values of Entropy, Chi-Squared Test p Value and KS Test p Value Across All Hybridization Algorithms [RQ 1, 2]

In [80]:
csv = pd.read_csv(TABLES + "/sequences_groupedby_hybrid.csv")
hybrids = ["none", "xor", "ilv","chn"]
halg_labels = ["None", "XOR Combination", "Conditional Interleaving", "Sequential Chaining"]
values = {
    "Entropy * 10^2": np.round(tuple(csv.set_index("hybrid type").loc[hybrids, "entropy"].astype(float) * 100), 0),
    "Chi-Squared Test Chi Value": np.round(tuple(csv.set_index("hybrid type").loc[hybrids, "chi-square test chi value"].astype(float)), 0),
    "KS Test D Value * 10^6": np.round(tuple(csv.set_index("hybrid type").loc[hybrids, "kolmogorov-smirnov test d value"].astype(float) * 1000000), 0),
}

x = np.arange(len(hybrids))
width = 0.25
multiplier = 0
fig, ax = plt.subplots(layout="constrained")
for attribute, measurement in values.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

ax.set_ylabel("Value")
ax.set_xticks(x + width, halg_labels)
ax.legend(loc="upper left", ncols=3)
ax.set_ylim(0, 1200)

plt.savefig(FIGURES + "/fig2.1.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

In [81]:
csv = pd.read_csv(TABLES + "/sequences_groupedby_hybrid.csv")
hybrids = ["none", "xor", "ilv", "wmx", "chn"]
halg_labels = ["None", "XOR Combination", "Conditional Interleaving", "Weighted Mixing", "Sequential Chaining"]
values = {
    "Entropy * 10^2": np.round(tuple(csv.set_index("hybrid type").loc[hybrids, "entropy"].astype(float) * 100), 0),
    "Chi-Squared Test Chi Value": np.round(tuple(csv.set_index("hybrid type").loc[hybrids, "chi-square test chi value"].astype(float)), 0),
    "KS Test D Value * 10^6": np.round(tuple(csv.set_index("hybrid type").loc[hybrids, "kolmogorov-smirnov test d value"].astype(float) * 1000000), 0),
}

x = np.arange(len(hybrids))
width = 0.25
multiplier = 0
fig, ax = plt.subplots(layout="constrained")
for attribute, measurement in values.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

ax.set_ylabel("Value")
ax.set_xticks(x + width, halg_labels)
ax.legend(loc="upper left", ncols=3)
ax.set_ylim(0, 180000)

plt.savefig(FIGURES + "/fig2.2.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Detected Repetition Distances Across All Sequences (Undetected Distances Treated as infinite) [RQ 3]

In [82]:
csv = pd.read_csv(TABLES + "/sequences_stats.csv")
names = csv["name"]
name_labels = [""] * len(names)
distances = csv["repetition distance"]

ylim = 160000
for d in range(len(distances)):
    if distances[d] == float('inf'):
        distances[d] = ylim + 1

x = range(len(names))

plt.bar(x, distances)
plt.xticks(x, name_labels)
plt.ylabel("Repetition Distance")
plt.xlabel("Sequences in Alphabetical Order by Devised Names")
plt.ylim(0, ylim)

plt.savefig(FIGURES + "/fig3.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Variations of Entropy, Chi-Squared Test p Value and KS Test p Value in Selected Samples Across Multiple Seeds [RQ 4]

In [83]:
csv = pd.read_csv(TABLES + "/samples_stats.csv")
fig, ax = plt.subplots(3, 1, sharex=True)
col_names = ["entropy", "chi-square test p value", "kolmogorov-smirnov test p value"]
plt_titles = ["Entropy", "Chi-Squared Test p Value", "KS Test p Value"]

for i in range(3):
    for sample in range(1, 8):
        df = csv[csv["name"].str.contains("sample" + str(sample))]
        values = df[col_names[i]]
        ax[i].plot(SAMPLE_SEEDS, values, label=str(sample) if i == 0 else None)
    ax[i].set_ylabel("Value")
    ax[i].set_title(plt_titles[i])

fig.legend(loc="lower center", ncols=7, bbox_to_anchor=(0.5, -0.07))
plt.xlabel("Seed")
plt.tight_layout()

plt.savefig(FIGURES + "/fig4.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Randomness Quality and Computational Efficiency Across All Possible PRNG + Hybrid Algorithm Combinations, Grouped By Hybrid Algorithms [RQ 5]

In [84]:
csv = pd.read_csv(TABLES + "/sequences_stats.csv")
csv_t = pd.read_csv(TABLES + "/generation_times.csv")
halg_names = ["none", "xor", "ilv", "wmx_a0.05", "wmx_a0.25", "wmx_a0.5", "wmx_a0.75", "wmx_a0.95", "chn"]
halg_labels = ["None", "XOR Combination", "Conditional Interleaving", "Weighted Mixing (alpha = 0.05)", "Weighted Mixing (alpha = 0.25)",
               "Weighted Mixing (alpha = 0.5)", "Weighted Mixing (alpha = 0.75)", "Weighted Mixing (alpha = 0.95)", "Sequential Chaining"]

for halg in halg_names:
    df = csv[csv["name"].str.contains(halg)]
    values = list(df["kolmogorov-smirnov test p value"])
    values = [str(i).replace(",", ".") for i in values] #changing format to python compatible floats
    values = [float(i) for i in values] #converting strings to floats
    df = csv_t[csv_t["name"].str.contains(halg)]
    throughputs = list(df["throughput"])
    throughputs = [i.replace(",", ".") for i in throughputs] #changing format to python compatible floats
    throughputs = [float(i) for i in throughputs] #converting strings to floats
    plt.scatter(values, throughputs, label=halg, alpha=0.5, s=80)

plt.legend(halg_labels, loc="lower center", ncols=3, bbox_to_anchor=(0.5, -0.25))
plt.xlabel("KS Test p Value")
plt.ylabel("Throughput (Numbers / Sec)")
plt.xlim(0, 1)

plt.savefig(FIGURES + "/fig5.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Accuracy of Pi Value Approximations Derived From Monte Carlo Pi Approximation Method [RQ 6]

In [85]:
csv = pd.read_csv(TABLES + "/sequences_stats.csv")
halg_names = ["none", "xor", "ilv", "wmx_a0.05", "wmx_a0.25", "wmx_a0.5", "wmx_a0.75", "wmx_a0.95", "chn"]
halg_labels = ["None", "XOR Combination", "Conditional Interleaving", "Weighted Mixing (alpha = 0.05)", "Weighted Mixing (alpha = 0.25)",
               "Weighted Mixing (alpha = 0.5)", "Weighted Mixing (alpha = 0.75)", "Weighted Mixing (alpha = 0.95)", "Sequential Chaining", "Pi"]

for halg in halg_names:
    df = csv[csv["name"].str.contains(halg)]
    values = list(df["pi approximation (monte carlo)"])
    values = [str(i).replace(",", ".") for i in values] #changing format to python compatible floats
    values = [float(i) for i in values] #converting strings to floats
    x = range(len(df))
    plt.plot(x, values, label=halg)

plt.plot(range(9), [np.pi]*9, label="Pi", c="k")
plt.legend(halg_labels, loc="lower center", ncols=2, bbox_to_anchor=(0.5, -0.35))
plt.ylabel("Value")
plt.xlabel("Sequence Ordinals for a Given Algorithm")

plt.savefig(FIGURES + "/fig6.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()

Autocorrelation and Bit Autocorrelation Values with Different Lags [RQ 7]

In [86]:
csv = pd.read_csv(TABLES + "/sequences_stats.csv")
fig, ax = plt.subplots(3, 2)
halg_names = ["none", "xor", "ilv", "wmx_a0.05", "wmx_a0.25", "wmx_a0.5", "wmx_a0.75", "wmx_a0.95", "chn"]
halg_labels = ["None", "XOR Combination", "Conditional Interleaving", "Weighted Mixing (alpha = 0.05)", "Weighted Mixing (alpha = 0.25)",
               "Weighted Mixing (alpha = 0.5)", "Weighted Mixing (alpha = 0.75)", "Weighted Mixing (alpha = 0.95)", "Sequential Chaining"]

for typ in range(2):
    for lag in range(3):
        for halg in halg_names:
            df = csv[csv["name"].str.contains(halg)]
            col = ""
            col += "bit " if typ == 1 else ""
            col += "autocorrelation (lag "
            match (typ, lag):
                case (0, 0): l = 1
                case (0, 1): l = 50
                case (0, 2): l = 200
                case (1, 0): l = 1
                case (1, 1): l = 15
                case (1, 2): l = 50
            col += str(l) + ")"
            values = df[col]
            values = [str(i).replace(",", ".") for i in values] #changing format to python compatible floats
            values = [float(i) for i in values] #converting strings to floats
            ax[lag][typ].plot(range(len(df)), values, label = halg if (lag, typ) == (0, 0) else None)
        ax[lag][typ].set_title(col)

fig.legend(halg_labels, loc="lower center", ncols=3, bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()

plt.savefig(FIGURES + "/fig7.pdf", bbox_inches="tight")
if show_figures: plt.show()
plt.close()